In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import ADJUSTMENT_SETS, OUTCOMES
from paths import paths

In [ ]:
OUT = experiment_paths('spatial/centre_periphery_adjustment')
DISEASES = OUTCOMES

data = load_data()
walkability_sd = data.walkability_score.std(ddof = 1)

In [ ]:
# Load the shared geographic inputs
data = gpd.read_parquet(paths.analysis.experiments['geography/gradients_and_shortfalls'].spatial_parquet)

gradient_summary = json.loads(
    (paths.analysis.experiments['geography/gradients_and_shortfalls'].summary_json).read_text()
)
gradient_formula = gradient_summary['gradient_formula']

In [ ]:
# Assess associations after accounting for centre–periphery structure
centrality_terms = '(seg1 + seg2 + seg3) * C(sector)'

centrality_rows = []
for adjustment, covariates in ADJUSTMENT_SETS.items():
    for outcome in DISEASES:
        model_data = data.loc[complete_case_mask(data, outcome, covariates)].copy()

        base_formula = f'{outcome} ~ walkability_score + {" + ".join(covariates)}'
        base = smf.ols(base_formula, data = model_data).fit()
        centrality = smf.ols(f'{base_formula} + {centrality_terms}', data = model_data).fit()
        
        residual_walkability_sd = smf.ols(gradient_formula, data = model_data).fit().resid.std(ddof = 1)

        base_coef = abs(base.params['walkability_score'])
        centrality_coef = abs(centrality.params['walkability_score'])
        attenuation_percent = 100 * (1 - centrality_coef / base_coef)

        for label, model, residual_sd in [
            ('Baseline', base, np.nan),
            ('Centre–periphery adjusted', centrality, residual_walkability_sd)
        ]:
            ci = model.conf_int().loc['walkability_score'] * walkability_sd

            centrality_rows.append(
                {
                    'adjustment': adjustment,
                    'outcome': outcome,
                    'model': label,
                    'n_lsoas': int(model.nobs),
                    'coefficient_per_sd': model.params['walkability_score'] * walkability_sd,
                    'coefficient_per_global_walkability_sd': model.params['walkability_score'] * walkability_sd,
                    'ci_lower_per_sd': ci.iloc[0],
                    'ci_upper_per_sd': ci.iloc[1],
                    'p_value': model.pvalues['walkability_score'],
                    'coefficient_per_residual_walkability_sd': model.params['walkability_score'] * residual_sd,
                    'residual_walkability_sd': residual_walkability_sd,
                    'r_squared': model.rsquared,
                    'attenuation_percent': attenuation_percent
                }
            )

centrality_results = pd.DataFrame(centrality_rows)

centrality_results.to_csv(OUT.centre_periphery_models_csv, index = False)

display(centrality_results.round(4))